## Part I

### Solution via for loop (slow)

In [124]:
def exchange_correlation_slow(A, B):
    
    X, Y, Z = A.shape # 1000 x 1000 x 30
    A_reshaped = A.reshape(X * Y, Z)
    X_reshaped = A_reshaped.shape[0]
    
    mean_corr_for_traces = []
    for i in range(X_reshaped):

        res = np.mean((A_reshaped[i] - A_reshaped[i].mean()) * (B - B.mean(axis=1)[:, None]), axis=1) /  (A_reshaped[i].std() * B.std(axis=1))
        mean_corr_for_traces.append(np.mean(res))
    
    return mean_corr_for_traces

### Solution via matrix mul (fast)

In [97]:
def exchange_correlation(A, B):
    """ For each trace in the trace matrix A computes mean correlation with each trace in trace matrix B
    
    Parameters 
    ----------
    A: np.ndarray, shape = (X, Y, Z)
        First trace matrix. Matrix A has X * Y traces.
    B: np.ndarray, shape = (N, Z)
        Second trace matrix.
        
    Returns
    -------
    correlation_vector: np.ndarray, shape = (X * Y,)
        Vector with mean values of correlation for each trace in A. 
    """
    X, Y, Z = A.shape # 1000 x 1000 x 30

    A_reshaped = A.reshape(X * Y, Z)
    
    # normalize matrices to compute correaltions
    A_normalized = (A_reshaped - A_reshaped.mean(axis=1)[:, None]) / A_reshaped.std(axis=1)[:, None]
    B_normalized = (B - B.mean(axis=1)[:, None]) / B.std(axis=1)[:, None]
    
    # compute correlation for each trace in A matrix (we have X * Y traces)
    correlation_vector = np.mean(1 / Z * A_normalized @ B_normalized.T, axis=1) 
    
    return correlation_vector

### Trials

In [125]:
#A = np.random.choice(1000, (1000, 1000, 30))
A = np.random.randn(1000, 1000, 30)

Slow solution with N = 20

In [128]:
%%time 
N = 20
B = np.random.randn(N, 30)
exchange_correlation_slow(A, B)

CPU times: user 1min 43s, sys: 91.9 ms, total: 1min 43s
Wall time: 1min 43s


[-0.013750176390702371,
 0.015190297894067572,
 -0.0194921603452931,
 -0.04003355608685995,
 0.017339694902027313,
 -0.026807645704629256,
 0.06559528838405992,
 0.03399252833867181,
 -0.002415877862783096,
 0.02731597242899798,
 -0.00430962411590859,
 0.010595083471099323,
 0.03185641545757752,
 0.035095371512138714,
 0.038181230010374344,
 0.01919340703198531,
 -0.03633943223280624,
 0.12882076921766933,
 0.030172386239004624,
 0.057031831145189414,
 -0.0103497044270729,
 0.06413168286816107,
 -0.009477945000741535,
 -0.017405306987843202,
 0.028614182379572244,
 -0.056994073413531174,
 -0.07883264055730554,
 0.023818992270369745,
 0.004090633831028439,
 -0.009118893032706228,
 0.006683335754364564,
 -0.005902677840090908,
 0.01065972559399029,
 -0.0024291620618256196,
 0.006984820233396795,
 -0.007354169320502773,
 -0.005065078422521796,
 -0.07095854920411702,
 -0.039676068244981225,
 0.0367632617890778,
 0.05173164882906598,
 0.018913992945224924,
 -0.05027947251865252,
 0.01285741

Fast solution with different N

In [129]:
%%time
N = 20 
B = np.random.randn(N, 30)
exchange_correlation(A, B)

CPU times: user 2.71 s, sys: 3.7 s, total: 6.42 s
Wall time: 555 ms


array([-0.11091381,  0.0309264 ,  0.02843721, ...,  0.04541236,
       -0.05368382, -0.02139417])

In [130]:
%%time
N = 50 
B = np.random.randn(N, 30)
exchange_correlation(A, B)

CPU times: user 8.46 s, sys: 4.75 s, total: 13.2 s
Wall time: 645 ms


array([-0.00181248,  0.0375764 , -0.0456612 , ...,  0.02219669,
       -0.03261392,  0.03865316])

In [131]:
%%time
N = 70 
B = np.random.randn(N, 30)
exchange_correlation(A, B)

CPU times: user 9.46 s, sys: 9.33 s, total: 18.8 s
Wall time: 747 ms


array([-0.01156153, -0.00078569, -0.01910763, ...,  0.00207403,
       -0.05925802,  0.00563509])

In [132]:
%%time
N = 100 
B = np.random.randn(N, 30)
exchange_correlation(A, B)

CPU times: user 12 s, sys: 11.3 s, total: 23.4 s
Wall time: 837 ms


array([ 0.00259923, -0.0227537 , -0.01154122, ..., -0.00207461,
        0.01106065, -0.02002099])

## Part II

### Slow solution 

In [2]:
def average_corr(curr_trace, K, curr_i, curr_j):
    """ Computes mean correaltion for each trace in the middle of the window with shape = (K_x, K_y). """
    K_x, K_y = K, K
    
    # consider border traces 
    if curr_i == 0 or curr_i == A.shape[0] - 1:
        K_y = K - 1
    
    if curr_j == 0 or curr_j == A.shape[1] - 1:
        K_x = K - 1
        
    sum_corr = 0   
    for i in range(0, K_x):
        for j in range(0, K_y):
            if i == curr_i and j == curr_j: # don't consider the central trace with itself 
                continue
            sum_corr += np.corrcoef(curr_trace, A[i, j])[0, 1]
            
    return sum_corr / (K_x * K_y - 1)

def window_correlation(A, K):
    """ Applies sliding window to i, j trace in trace matrix A. """
    X, Y = A.shape[0], A.shape[1]
    output = np.zeros((X, Y))
    
    for i in range(X):
        for j in range(Y):
            output[i, j] = average_corr(A[i, j], K, i, j)
    
    return output 

### Fast solution

In [1]:
def trace_correlation(A, K):
    """ For window KxK computes mean correlation for trace in the middle of the window and its neighbours. 
    
    Parameters
    ----------
    A: np.ndarray, shape = (X, Y, Z)
        Trace matrix
    K: int, odd number
        Size of the kernel 
        
    Returns 
    -------
    mean_corrs: np.ndarray, shape = (X, Y)
       Matrix with mean correlation for each trace in the i, j position   
    """
    # other parameters
    padding = K // 2 
    mid = K ** 2 // 2 # idx of mid trace 
    X, Y, Z = A.shape # 1000 x 1000 x 30 
    
    # add padding to the trace matrix 
    A_pad = np.pad(A, pad_width=((padding, padding), (padding, padding), (0, 0)))
    
    # make K x K x Z patches
    patches = np.lib.stride_tricks.sliding_window_view(A_pad, window_shape=(K, K, Z))
    patches_reshaped = patches.reshape(patches.shape[0] * patches.shape[1], patches.shape[2] * patches.shape[3] * patches.shape[4], patches.shape[5])
    
    # normalize and transpose patches to compute correlations
    patches_norm = (patches_reshaped - patches_reshaped.mean(axis=2)[:, :, None]) / patches_reshaped.std(axis=2)[:, :, None] 
    patches_norm_reshaped = patches_norm.transpose((0, 2, 1))
    
    # find correlations for each trace in each patch and choose only mid trace
    cor_matrix = 1 / Z * (patches_norm @ patches_norm_reshaped)[:, mid, :]
    cor_matrix[:, mid] = np.nan # replace ones to nans so that don't consider trace with itslef  
    mean_corrs = np.nanmean(cor_matrix, axis=1).reshape(X, Y) # compute average correlation for each trace
    
    return mean_corrs

### Trials

In [1721]:
#A = np.random.choice(1000, (1000, 1000, 30))

In [81]:
A = np.random.randn(1000, 1000, 30)

Slow solution (K=3)

In [89]:
%%time
K = 3
window_correlation(A, K)

CPU times: user 18min 23s, sys: 19.9 s, total: 18min 43s
Wall time: 18min 24s


array([[ 0.13844292, -0.00628207, -0.06038281, ...,  0.02067526,
        -0.11688274,  0.00114831],
       [-0.00091623, -0.0398798 , -0.02607964, ...,  0.00339347,
         0.03157959, -0.04082064],
       [-0.0520622 , -0.01495977,  0.04393026, ...,  0.0728345 ,
         0.01231182, -0.05547083],
       ...,
       [ 0.0150817 , -0.07673415, -0.06854009, ..., -0.08250177,
        -0.03751466,  0.00369667],
       [ 0.03342074,  0.09616385, -0.07580428, ..., -0.03453307,
        -0.09703197, -0.17995682],
       [ 0.09948376,  0.18956556, -0.10480127, ..., -0.0099343 ,
         0.05670798, -0.01056972]])

Fast solution (K from 3 to 13)

In [82]:
%%time
K = 3
trace_correlation(A=A, K=K)

/tmp/ipykernel_51127/2529112910.py:29: RuntimeWarning: invalid value encountered in true_divide
  patches_norm = (patches_reshaped - patches_reshaped.mean(axis=2)[:, :, None]) / patches_reshaped.std(axis=2)[:, :, None]


CPU times: user 2.93 s, sys: 1.91 s, total: 4.84 s
Wall time: 4.83 s


array([[ 0.10383219, -0.03561156,  0.05267085, ...,  0.0649684 ,
        -0.08087632, -0.0368375 ],
       [ 0.0250089 , -0.03544871, -0.0460357 , ...,  0.05304436,
        -0.04240341, -0.00739723],
       [-0.05106949,  0.0051978 ,  0.04749235, ...,  0.0893184 ,
         0.06096202, -0.04533566],
       ...,
       [ 0.00397553, -0.02752037,  0.03306454, ...,  0.06919086,
         0.12531035, -0.00195976],
       [-0.02368721,  0.00650822, -0.04650503, ...,  0.04164942,
         0.01588255,  0.08432181],
       [-0.03290236, -0.02652787, -0.10443953, ..., -0.15194395,
        -0.05056561, -0.02742862]])

In [83]:
%%time
K = 5
trace_correlation(A=A, K=K)

/tmp/ipykernel_51127/2529112910.py:29: RuntimeWarning: invalid value encountered in true_divide
  patches_norm = (patches_reshaped - patches_reshaped.mean(axis=2)[:, :, None]) / patches_reshaped.std(axis=2)[:, :, None]


CPU times: user 8.42 s, sys: 5.47 s, total: 13.9 s
Wall time: 13.9 s


array([[ 0.04716586,  0.00310675, -0.03226459, ...,  0.00270904,
         0.00379276, -0.01394974],
       [-0.03222013,  0.01433287, -0.04091293, ...,  0.02271155,
        -0.01161909, -0.05662789],
       [-0.05225524,  0.00754491,  0.00737262, ...,  0.04185461,
        -0.00131972, -0.00533415],
       ...,
       [-0.006995  , -0.00836229,  0.02308557, ...,  0.01075162,
         0.0436353 ,  0.03182146],
       [-0.05133023, -0.05543476, -0.03510281, ...,  0.04952032,
         0.01720932, -0.00831725],
       [ 0.00092814, -0.05854078,  0.01369805, ..., -0.04302839,
        -0.02408667,  0.00164122]])

In [84]:
%%time
K = 7
trace_correlation(A=A, K=K)

/tmp/ipykernel_51127/2529112910.py:29: RuntimeWarning: invalid value encountered in true_divide
  patches_norm = (patches_reshaped - patches_reshaped.mean(axis=2)[:, :, None]) / patches_reshaped.std(axis=2)[:, :, None]


CPU times: user 17.5 s, sys: 11.8 s, total: 29.3 s
Wall time: 29.3 s


array([[ 0.00434347,  0.00664675, -0.04012006, ..., -0.01676317,
        -0.04462925, -0.018159  ],
       [ 0.00659062, -0.0093722 , -0.00713689, ...,  0.00662771,
         0.03695038,  0.01544248],
       [-0.00639145,  0.03386906,  0.02948491, ...,  0.01654382,
        -0.01583627,  0.00318748],
       ...,
       [-0.00943125, -0.02432525,  0.01338893, ..., -0.00177488,
         0.04997491,  0.010258  ],
       [-0.04612621, -0.03596669, -0.04368645, ...,  0.01641053,
         0.04225861, -0.02338424],
       [ 0.06636108, -0.00726538,  0.01848947, ..., -0.0582047 ,
        -0.02869619,  0.00729025]])

In [85]:
%%time
K = 9
trace_correlation(A=A, K=K)

/tmp/ipykernel_51127/2529112910.py:29: RuntimeWarning: invalid value encountered in true_divide
  patches_norm = (patches_reshaped - patches_reshaped.mean(axis=2)[:, :, None]) / patches_reshaped.std(axis=2)[:, :, None]


CPU times: user 33.6 s, sys: 23.6 s, total: 57.2 s
Wall time: 57.1 s


array([[ 0.0591288 ,  0.0259437 , -0.05060857, ...,  0.01172908,
        -0.00714064,  0.01363872],
       [ 0.02436567, -0.02801795, -0.00382056, ...,  0.03967607,
         0.04118635,  0.01987042],
       [-0.01173542,  0.0276796 ,  0.01689771, ...,  0.03266082,
         0.00598913, -0.01910688],
       ...,
       [ 0.00513344, -0.02599562,  0.00542944, ..., -0.00845736,
         0.03794984, -0.01355203],
       [-0.07368486,  0.01070014, -0.01639929, ...,  0.02007661,
         0.03063257, -0.02648518],
       [ 0.03836974,  0.01843552,  0.02329052, ..., -0.01479685,
        -0.01258709, -0.01906448]])

In [86]:
%%time
K = 11
trace_correlation(A=A, K=K)

/tmp/ipykernel_51127/2529112910.py:29: RuntimeWarning: invalid value encountered in true_divide
  patches_norm = (patches_reshaped - patches_reshaped.mean(axis=2)[:, :, None]) / patches_reshaped.std(axis=2)[:, :, None]


CPU times: user 53.6 s, sys: 44.9 s, total: 1min 38s
Wall time: 1min 38s


array([[ 0.05976701,  0.00377235, -0.04343497, ...,  0.00959974,
        -0.03923696, -0.00127864],
       [ 0.01237268, -0.00585729, -0.02432412, ...,  0.04186357,
         0.02620113,  0.0163093 ],
       [-0.00376981,  0.02075106,  0.00102352, ...,  0.0057035 ,
         0.00258482, -0.03313537],
       ...,
       [-0.00767072, -0.05215122,  0.02858629, ..., -0.00290172,
         0.01698981, -0.02410105],
       [-0.03680877,  0.01715711, -0.00449636, ..., -0.0027261 ,
        -0.00121413, -0.01522229],
       [-0.00979724,  0.01332555,  0.03147126, ...,  0.00642591,
        -0.02450767, -0.00614348]])

In [87]:
%%time
K = 13
trace_correlation(A=A, K=K)

/tmp/ipykernel_51127/2529112910.py:29: RuntimeWarning: invalid value encountered in true_divide
  patches_norm = (patches_reshaped - patches_reshaped.mean(axis=2)[:, :, None]) / patches_reshaped.std(axis=2)[:, :, None]


CPU times: user 4min 39s, sys: 9min 41s, total: 14min 21s
Wall time: 3min 22s


array([[ 0.03326139, -0.00978646, -0.02943666, ...,  0.0002634 ,
        -0.02573096, -0.00833081],
       [ 0.00918568,  0.00800224, -0.021117  , ...,  0.00316729,
         0.04148731,  0.01827915],
       [ 0.00833279,  0.01596544, -0.00765911, ...,  0.00052807,
        -0.00016465, -0.03059127],
       ...,
       [-0.0057139 , -0.03616805,  0.00384784, ..., -0.00590482,
        -0.00734617, -0.00516221],
       [-0.04051934,  0.00495578, -0.00393329, ..., -0.00299269,
        -0.0079942 , -0.02459485],
       [ 0.00023077,  0.02866945,  0.00667845, ..., -0.00184908,
        -0.03143762, -0.01217306]])